In [2]:
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm


# ============================================================
# Configuration
# ============================================================

CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/boneage-training-dataset.csv"

IMAGE_DIR = "/content/drive/MyDrive/Colab Notebooks/cropped_overlayed_RSNA_dataset"

BEST_CHECKPOINT_PATH = "/content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_best.pth"
LATEST_CHECKPOINT_PATH = "/content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_latest.pth"
PLOT_DIR = "/content/drive/MyDrive/Colab Notebooks/training_plots_efficientnet_b4_512x512"

RESUME_TRAINING = True
RESUME_FROM_BEST_IF_NO_LATEST = True

MODEL_NAME = "tf_efficientnet_b4.ns_jft_in1k"

IMAGE_HEIGHT = 512
IMAGE_WIDTH = 512

NO_PRETRAINED = False

VAL_SIZE = 0.15
SEED = 42

BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 2

EPOCHS = 100
WARMUP_EPOCHS = 5
PATIENCE = 15

BACKBONE_LR = 1e-4
HEAD_LR = 5e-4
WEIGHT_DECAY = 1e-5

DROP_PATH = 0.1
HEAD_DROPOUT = 0.2
HIDDEN_DIM = 512

SMOOTH_L1_BETA = 6.0
MAX_GRAD_NORM = 1.0

USE_AMP = True

SUPPORTED_IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ============================================================
# Image indexing / CSV filtering
# ============================================================

def normalize_id(value) -> str:
    if pd.isna(value):
        return ""

    if isinstance(value, float) and value.is_integer():
        return str(int(value))

    return str(value).strip()


def build_image_index(image_dir):
    image_dir = Path(image_dir)

    if not image_dir.exists():
        raise FileNotFoundError(f"IMAGE_DIR does not exist: {image_dir}")

    image_index = {}

    for ext in SUPPORTED_IMAGE_EXTENSIONS:
        for path in image_dir.glob(f"*{ext}"):
            image_index[path.stem] = path

    return image_index


def filter_dataframe_to_existing_images(df, image_index, plot_dir):
    df = df.copy()
    df["id"] = df["id"].apply(normalize_id)

    exists_mask = df["id"].isin(image_index.keys())
    missing_df = df.loc[~exists_mask].copy()
    filtered_df = df.loc[exists_mask].copy()

    plot_dir = Path(plot_dir)
    plot_dir.mkdir(parents=True, exist_ok=True)

    if len(missing_df) > 0:
        missing_path = plot_dir / "missing_images.csv"
        missing_df.to_csv(missing_path, index=False)

        print(f"Warning: {len(missing_df)} rows were removed because no image file was found.")
        print(f"Missing image IDs saved to: {missing_path}")
        print("First missing IDs:", missing_df["id"].head(20).tolist())
    else:
        print("All CSV rows have matching image files.")

    print(f"Samples after image-file filtering: {len(filtered_df)}")

    return filtered_df


# ============================================================
# Dataset
# ============================================================

class BoneAgeDataset(Dataset):
    def __init__(self, dataframe, image_index, image_height, image_width):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_index = image_index

        self.transform = transforms.Compose([
            transforms.Resize((image_height, image_width)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_id = normalize_id(row["id"])
        image_path = self.image_index[image_id]

        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)

        boneage = torch.tensor(float(row["boneage"]), dtype=torch.float32)

        male_value = row["male"]
        if isinstance(male_value, str):
            male_value = male_value.lower() == "true"

        male = torch.tensor([float(male_value)], dtype=torch.float32)

        return {
            "image": image,
            "male": male,
            "target": boneage,
            "id": image_id,
        }


# ============================================================
# Model
# ============================================================

class BoneAgeEfficientNet(nn.Module):
    def __init__(
        self,
        model_name: str,
        pretrained: bool = True,
        drop_path_rate: float = 0.1,
        head_dropout: float = 0.2,
        hidden_dim: int = 512,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
            drop_path_rate=drop_path_rate,
        )

        feature_dim = self.backbone.num_features

        self.regression_head = nn.Sequential(
            nn.Linear(feature_dim + 1, hidden_dim),
            nn.SiLU(),
            nn.Dropout(head_dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, image, male):
        features = self.backbone(image)
        features = torch.cat([features, male], dim=1)
        prediction = self.regression_head(features).squeeze(1)
        return prediction


# ============================================================
# Split
# ============================================================

def create_stratified_split(df, val_size, seed):
    df = df.copy()

    df["age_bin"] = pd.qcut(
        df["boneage"],
        q=10,
        duplicates="drop",
        labels=False,
    )

    df["male_str"] = df["male"].astype(str)
    df["stratify_col"] = df["age_bin"].astype(str) + "_" + df["male_str"]

    stratify_col = df["stratify_col"]

    if stratify_col.value_counts().min() < 2:
        print(
            "Warning: Some age+gender strata contain fewer than 2 samples. "
            "Falling back to stratification by age_bin only."
        )
        stratify_col = df["age_bin"]

    if pd.Series(stratify_col).value_counts().min() < 2:
        print(
            "Warning: Some age bins contain fewer than 2 samples. "
            "Falling back to random split."
        )
        stratify_col = None

    train_df, val_df = train_test_split(
        df,
        test_size=val_size,
        random_state=seed,
        shuffle=True,
        stratify=stratify_col,
    )

    train_df = train_df.drop(columns=["age_bin", "male_str", "stratify_col"])
    val_df = val_df.drop(columns=["age_bin", "male_str", "stratify_col"])

    return train_df, val_df


# ============================================================
# Optimizer
# ============================================================

def build_optimizer(model, backbone_lr, head_lr, weight_decay):
    decay_params = []
    no_decay_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if name.startswith("regression_head"):
            head_params.append(param)
        elif param.ndim < 2 or name.endswith(".bias"):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": decay_params,
                "lr": backbone_lr,
                "weight_decay": weight_decay,
            },
            {
                "params": no_decay_params,
                "lr": backbone_lr,
                "weight_decay": 0.0,
            },
            {
                "params": head_params,
                "lr": head_lr,
                "weight_decay": weight_decay,
            },
        ]
    )

    return optimizer


# ============================================================
# Scheduler
# ============================================================

def build_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(max(1, warmup_epochs))

        progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ============================================================
# Metrics
# ============================================================

def compute_mae(preds, targets):
    preds = np.asarray(preds)
    targets = np.asarray(targets)
    return float(np.mean(np.abs(preds - targets)))


def compute_rmse(preds, targets):
    preds = np.asarray(preds)
    targets = np.asarray(targets)
    return float(np.sqrt(np.mean((preds - targets) ** 2)))


def compute_median_absolute_error(preds, targets):
    preds = np.asarray(preds)
    targets = np.asarray(targets)
    return float(np.median(np.abs(preds - targets)))


def compute_mean_error(preds, targets):
    preds = np.asarray(preds)
    targets = np.asarray(targets)
    return float(np.mean(preds - targets))


def group_metrics(preds, targets, males):
    df = pd.DataFrame({
        "pred": preds,
        "target": targets,
        "male": males,
    })

    df["abs_error"] = (df["pred"] - df["target"]).abs()

    age_bins = [0, 24, 48, 72, 96, 120, 144, 168, 192, 240]
    df["age_group"] = pd.cut(
        df["target"],
        bins=age_bins,
        right=False,
        include_lowest=True,
    )

    age_metrics = (
        df.groupby("age_group", observed=False)["abs_error"]
        .agg(["count", "mean", "median"])
        .reset_index()
    )

    gender_metrics = (
        df.groupby("male")["abs_error"]
        .agg(["count", "mean", "median"])
        .reset_index()
    )

    return age_metrics, gender_metrics


# ============================================================
# Train / Validate
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    scaler,
    use_amp,
    grad_accum_steps,
    max_grad_norm,
):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_targets = []

    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for step, batch in enumerate(progress_bar):
        images = batch["image"].to(device, non_blocking=True)
        males = batch["male"].to(device, non_blocking=True)
        targets = batch["target"].to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=use_amp and device.type == "cuda",
        ):
            preds = model(images, males)
            loss = criterion(preds, targets)
            loss_for_backward = loss / grad_accum_steps

        if scaler is not None:
            scaler.scale(loss_for_backward).backward()
        else:
            loss_for_backward.backward()

        should_step = (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader)

        if should_step:
            if scaler is not None:
                scaler.unscale_(optimizer)

            if max_grad_norm is not None and max_grad_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * images.size(0)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(targets.detach().cpu().numpy().tolist())

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "mae": f"{compute_mae(all_preds, all_targets):.2f}",
        })

    epoch_loss = running_loss / len(loader.dataset)
    epoch_mae = compute_mae(all_preds, all_targets)

    return epoch_loss, epoch_mae


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, use_amp):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_targets = []
    all_males = []
    all_ids = []

    progress_bar = tqdm(loader, desc="Validation", leave=False)

    for batch in progress_bar:
        images = batch["image"].to(device, non_blocking=True)
        males = batch["male"].to(device, non_blocking=True)
        targets = batch["target"].to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=use_amp and device.type == "cuda",
        ):
            preds = model(images, males)
            loss = criterion(preds, targets)

        running_loss += loss.item() * images.size(0)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(targets.detach().cpu().numpy().tolist())
        all_males.extend(males.detach().cpu().numpy().reshape(-1).tolist())
        all_ids.extend(batch["id"])

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "mae": f"{compute_mae(all_preds, all_targets):.2f}",
        })

    val_loss = running_loss / len(loader.dataset)
    val_mae = compute_mae(all_preds, all_targets)
    val_rmse = compute_rmse(all_preds, all_targets)
    val_median_ae = compute_median_absolute_error(all_preds, all_targets)
    val_mean_error = compute_mean_error(all_preds, all_targets)

    predictions_df = pd.DataFrame({
        "id": all_ids,
        "target": all_targets,
        "prediction": all_preds,
        "male": all_males,
        "error": np.asarray(all_preds) - np.asarray(all_targets),
        "abs_error": np.abs(np.asarray(all_preds) - np.asarray(all_targets)),
    })

    return val_loss, val_mae, val_rmse, val_median_ae, val_mean_error, predictions_df


# ============================================================
# Checkpointing
# ============================================================

def save_checkpoint(
    checkpoint_path,
    epoch,
    model,
    optimizer,
    scheduler,
    best_val_mae,
    best_epoch,
    patience_counter,
    history,
    config,
):
    checkpoint = {
        "epoch": epoch,
        "model_name": MODEL_NAME,
        "image_height": IMAGE_HEIGHT,
        "image_width": IMAGE_WIDTH,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_mae": best_val_mae,
        "best_epoch": best_epoch,
        "patience_counter": patience_counter,
        "history": history,
        "config": config,
    }

    torch.save(checkpoint, checkpoint_path)


def load_history_from_csv_if_available():
    history_path = Path(PLOT_DIR) / "history.csv"

    if history_path.exists():
        history_df = pd.read_csv(history_path)
        history = history_df.to_dict("records")
        print(f"Loaded history from: {history_path}")
        print(f"History entries loaded: {len(history)}")
        return history

    print("No history found in checkpoint or history.csv. History will continue from resume point.")
    return []


def resume_from_checkpoint_if_available(
    model,
    optimizer,
    scheduler,
    device,
):
    start_epoch = 1
    history = []
    best_val_mae = float("inf")
    best_epoch = -1
    patience_counter = 0

    latest_path = Path(LATEST_CHECKPOINT_PATH)
    best_path = Path(BEST_CHECKPOINT_PATH)

    checkpoint_path = None

    if RESUME_TRAINING and latest_path.exists():
        checkpoint_path = latest_path
        print(f"Resuming from latest checkpoint: {checkpoint_path}")
    elif RESUME_TRAINING and RESUME_FROM_BEST_IF_NO_LATEST and best_path.exists():
        checkpoint_path = best_path
        print(f"Latest checkpoint not found. Resuming from best checkpoint: {checkpoint_path}")
    else:
        print("No checkpoint loaded. Starting from scratch.")
        return start_epoch, history, best_val_mae, best_epoch, patience_counter

    checkpoint = torch.load(checkpoint_path, map_location=device)

    checkpoint_model_name = checkpoint.get("model_name", None)
    if checkpoint_model_name is not None and checkpoint_model_name != MODEL_NAME:
        raise ValueError(
            f"Checkpoint model_name does not match current MODEL_NAME.\n"
            f"Checkpoint: {checkpoint_model_name}\n"
            f"Current:    {MODEL_NAME}\n"
            f"Use a different checkpoint path or set RESUME_TRAINING = False."
        )

    checkpoint_image_height = checkpoint.get("image_height", None)
    checkpoint_image_width = checkpoint.get("image_width", None)

    if checkpoint_image_height is not None and checkpoint_image_height != IMAGE_HEIGHT:
        raise ValueError(
            f"Checkpoint image_height does not match current IMAGE_HEIGHT.\n"
            f"Checkpoint: {checkpoint_image_height}\n"
            f"Current:    {IMAGE_HEIGHT}\n"
            f"Use a different checkpoint path or set RESUME_TRAINING = False."
        )

    if checkpoint_image_width is not None and checkpoint_image_width != IMAGE_WIDTH:
        raise ValueError(
            f"Checkpoint image_width does not match current IMAGE_WIDTH.\n"
            f"Checkpoint: {checkpoint_image_width}\n"
            f"Current:    {IMAGE_WIDTH}\n"
            f"Use a different checkpoint path or set RESUME_TRAINING = False."
        )

    model.load_state_dict(checkpoint["model_state_dict"])

    if "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    else:
        print("Warning: optimizer_state_dict not found. Optimizer starts fresh.")

    if "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    else:
        print("Warning: scheduler_state_dict not found. Scheduler starts fresh.")

    checkpoint_epoch = int(checkpoint.get("epoch", 0))
    start_epoch = checkpoint_epoch + 1

    best_val_mae = float(checkpoint.get("best_val_mae", float("inf")))
    best_epoch = int(checkpoint.get("best_epoch", checkpoint_epoch))
    patience_counter = int(checkpoint.get("patience_counter", 0))

    history = checkpoint.get("history", [])

    if len(history) == 0:
        history = load_history_from_csv_if_available()

    print(f"Checkpoint epoch: {checkpoint_epoch}")
    print(f"Next epoch: {start_epoch}")
    print(f"Best validation MAE so far: {best_val_mae:.4f} months")
    print(f"Best epoch so far: {best_epoch}")
    print(f"Patience counter: {patience_counter}")

    return start_epoch, history, best_val_mae, best_epoch, patience_counter


# ============================================================
# Plotting
# ============================================================

def save_plots(history_df, predictions_df, plot_dir, prefix="latest"):
    plot_dir = Path(plot_dir)
    plot_dir.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], label="Train loss")
    axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], label="Val loss")
    axes[0, 0].set_title("Loss curves")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("SmoothL1 loss")
    axes[0, 0].legend()

    axes[0, 1].plot(history_df["epoch"], history_df["train_mae"], label="Train MAE")
    axes[0, 1].plot(history_df["epoch"], history_df["val_mae"], label="Val MAE")
    axes[0, 1].set_title("MAE curves")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("MAE in months")
    axes[0, 1].legend()

    axes[1, 0].scatter(
        predictions_df["target"],
        predictions_df["prediction"],
        alpha=0.5,
        s=12,
    )

    min_age = min(predictions_df["target"].min(), predictions_df["prediction"].min())
    max_age = max(predictions_df["target"].max(), predictions_df["prediction"].max())

    axes[1, 0].plot([min_age, max_age], [min_age, max_age], linestyle="--")
    axes[1, 0].set_title("Validation predictions")
    axes[1, 0].set_xlabel("True bone age in months")
    axes[1, 0].set_ylabel("Predicted bone age in months")

    age_bins = [0, 24, 48, 72, 96, 120, 144, 168, 192, 240]
    predictions_df = predictions_df.copy()
    predictions_df["age_group"] = pd.cut(
        predictions_df["target"],
        bins=age_bins,
        right=False,
        include_lowest=True,
    )

    group_mae = (
        predictions_df.groupby("age_group", observed=False)["abs_error"]
        .mean()
        .reset_index()
    )

    axes[1, 1].bar(
        group_mae["age_group"].astype(str),
        group_mae["abs_error"],
    )
    axes[1, 1].set_title("Validation MAE by age group")
    axes[1, 1].set_xlabel("Age group in months")
    axes[1, 1].set_ylabel("MAE in months")
    axes[1, 1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.savefig(plot_dir / f"{prefix}_training_diagnostics.png", dpi=200)
    plt.close()

    residuals = predictions_df["prediction"] - predictions_df["target"]

    plt.figure(figsize=(8, 5))
    plt.hist(residuals, bins=40)
    plt.title("Validation residuals")
    plt.xlabel("Prediction error in months")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(plot_dir / f"{prefix}_validation_residuals.png", dpi=200)
    plt.close()


def save_bland_altman_plot(predictions_df, plot_dir, prefix="latest"):
    plot_dir = Path(plot_dir)
    plot_dir.mkdir(parents=True, exist_ok=True)

    predictions = predictions_df["prediction"].values.astype(float)
    targets = predictions_df["target"].values.astype(float)

    mean_values = (predictions + targets) / 2.0
    errors = predictions - targets

    bias = float(np.mean(errors))
    sd_error = float(np.std(errors, ddof=1))

    lower_limit = float(bias - 1.96 * sd_error)
    upper_limit = float(bias + 1.96 * sd_error)

    plt.figure(figsize=(8, 6))
    plt.scatter(mean_values, errors, alpha=0.5, s=12)

    plt.axhline(bias, linestyle="-", label=f"Bias: {bias:.2f} months")
    plt.axhline(upper_limit, linestyle="--", label=f"+1.96 SD: {upper_limit:.2f}")
    plt.axhline(lower_limit, linestyle="--", label=f"-1.96 SD: {lower_limit:.2f}")
    plt.axhline(0, linestyle=":", label="Zero error")

    plt.title("Bland–Altman Plot")
    plt.xlabel("Mean of predicted and true bone age in months")
    plt.ylabel("Prediction error in months")
    plt.legend()
    plt.tight_layout()

    plt.savefig(plot_dir / f"{prefix}_bland_altman_plot.png", dpi=200)
    plt.close()

    bland_altman_stats = {
        "bias_mean_error": bias,
        "sd_error": sd_error,
        "lower_95_limit_of_agreement": lower_limit,
        "upper_95_limit_of_agreement": upper_limit,
    }

    pd.DataFrame([bland_altman_stats]).to_csv(
        plot_dir / f"{prefix}_bland_altman_stats.csv",
        index=False,
    )

    return bland_altman_stats


# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)

    plot_dir = Path(PLOT_DIR)
    plot_dir.mkdir(parents=True, exist_ok=True)

    best_checkpoint_path = Path(BEST_CHECKPOINT_PATH)
    latest_checkpoint_path = Path(LATEST_CHECKPOINT_PATH)

    best_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    latest_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    image_index = build_image_index(IMAGE_DIR)
    print(f"Found image files: {len(image_index)}")

    df = pd.read_csv(CSV_PATH)

    required_columns = {"id", "boneage", "male"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(f"CSV is missing required columns: {missing_columns}")

    df["id"] = df["id"].apply(normalize_id)

    print(f"Total CSV samples: {len(df)}")
    print(f"Bone age range: {df['boneage'].min()} to {df['boneage'].max()} months")
    print(f"Male distribution:\n{df['male'].value_counts()}")

    df = filter_dataframe_to_existing_images(
        df=df,
        image_index=image_index,
        plot_dir=plot_dir,
    )

    train_df, val_df = create_stratified_split(
        df=df,
        val_size=VAL_SIZE,
        seed=SEED,
    )

    print(f"Train samples: {len(train_df)}")
    print(f"Validation samples: {len(val_df)}")

    train_df.to_csv(plot_dir / "train_split.csv", index=False)
    val_df.to_csv(plot_dir / "val_split.csv", index=False)

    train_dataset = BoneAgeDataset(
        dataframe=train_df,
        image_index=image_index,
        image_height=IMAGE_HEIGHT,
        image_width=IMAGE_WIDTH,
    )

    val_dataset = BoneAgeDataset(
        dataframe=val_df,
        image_index=image_index,
        image_height=IMAGE_HEIGHT,
        image_width=IMAGE_WIDTH,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )

    model = BoneAgeEfficientNet(
        model_name=MODEL_NAME,
        pretrained=not NO_PRETRAINED,
        drop_path_rate=DROP_PATH,
        head_dropout=HEAD_DROPOUT,
        hidden_dim=HIDDEN_DIM,
    )

    model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Model: {MODEL_NAME}")
    print(f"Image size: {IMAGE_WIDTH}x{IMAGE_HEIGHT} width x height")
    print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    criterion = nn.SmoothL1Loss(beta=SMOOTH_L1_BETA)

    optimizer = build_optimizer(
        model=model,
        backbone_lr=BACKBONE_LR,
        head_lr=HEAD_LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = build_warmup_cosine_scheduler(
        optimizer=optimizer,
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=EPOCHS,
    )

    use_amp = USE_AMP and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    config = {
        "csv_path": CSV_PATH,
        "image_dir": IMAGE_DIR,
        "best_checkpoint_path": BEST_CHECKPOINT_PATH,
        "latest_checkpoint_path": LATEST_CHECKPOINT_PATH,
        "plot_dir": PLOT_DIR,
        "resume_training": RESUME_TRAINING,
        "resume_from_best_if_no_latest": RESUME_FROM_BEST_IF_NO_LATEST,
        "model_name": MODEL_NAME,
        "image_height": IMAGE_HEIGHT,
        "image_width": IMAGE_WIDTH,
        "no_pretrained": NO_PRETRAINED,
        "val_size": VAL_SIZE,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": BATCH_SIZE * GRAD_ACCUM_STEPS,
        "num_workers": NUM_WORKERS,
        "epochs": EPOCHS,
        "warmup_epochs": WARMUP_EPOCHS,
        "patience": PATIENCE,
        "backbone_lr": BACKBONE_LR,
        "head_lr": HEAD_LR,
        "weight_decay": WEIGHT_DECAY,
        "drop_path": DROP_PATH,
        "head_dropout": HEAD_DROPOUT,
        "hidden_dim": HIDDEN_DIM,
        "smooth_l1_beta": SMOOTH_L1_BETA,
        "max_grad_norm": MAX_GRAD_NORM,
        "use_amp": USE_AMP,
        "device": str(device),
        "total_params": total_params,
        "trainable_params": trainable_params,
        "found_image_files": len(image_index),
        "training_samples_after_filtering": len(df),
    }

    with open(plot_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    start_epoch, history, best_val_mae, best_epoch, patience_counter = resume_from_checkpoint_if_available(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
    )

    if start_epoch > EPOCHS:
        print(f"Checkpoint already reached epoch {start_epoch - 1}. EPOCHS is set to {EPOCHS}.")
        print("Increase EPOCHS if you want to continue training further.")
        return

    predictions_df = None

    for epoch in range(start_epoch, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")

        train_loss, train_mae = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            use_amp=use_amp,
            grad_accum_steps=GRAD_ACCUM_STEPS,
            max_grad_norm=MAX_GRAD_NORM,
        )

        val_loss, val_mae, val_rmse, val_median_ae, val_mean_error, predictions_df = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            use_amp=use_amp,
        )

        scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]

        epoch_log = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_mae": train_mae,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse,
            "val_median_ae": val_median_ae,
            "val_mean_error": val_mean_error,
            "lr": current_lr,
        }

        history.append(epoch_log)

        print(
            f"Train loss: {train_loss:.4f} | "
            f"Train MAE: {train_mae:.2f} months | "
            f"Val loss: {val_loss:.4f} | "
            f"Val MAE: {val_mae:.2f} months | "
            f"Val RMSE: {val_rmse:.2f} months | "
            f"Val Median AE: {val_median_ae:.2f} months | "
            f"Val Mean Error: {val_mean_error:.2f} months | "
            f"LR: {current_lr:.2e}"
        )

        history_df = pd.DataFrame(history)
        history_df.to_csv(plot_dir / "history.csv", index=False)

        predictions_df.to_csv(plot_dir / "latest_val_predictions.csv", index=False)

        # Save latest plots after every epoch.
        # This is useful if training is interrupted.
        save_plots(
            history_df=history_df,
            predictions_df=predictions_df,
            plot_dir=plot_dir,
            prefix="latest",
        )

        latest_bland_altman_stats = save_bland_altman_plot(
            predictions_df=predictions_df,
            plot_dir=plot_dir,
            prefix="latest",
        )

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_epoch = epoch
            patience_counter = 0

            save_checkpoint(
                checkpoint_path=best_checkpoint_path,
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                best_val_mae=best_val_mae,
                best_epoch=best_epoch,
                patience_counter=patience_counter,
                history=history,
                config=config,
            )

            predictions_df.to_csv(plot_dir / "best_val_predictions.csv", index=False)

            # Save best plots whenever a new best checkpoint is found.
            save_plots(
                history_df=history_df,
                predictions_df=predictions_df,
                plot_dir=plot_dir,
                prefix="best",
            )

            best_bland_altman_stats = save_bland_altman_plot(
                predictions_df=predictions_df,
                plot_dir=plot_dir,
                prefix="best",
            )

            print(f"Saved new best model to: {best_checkpoint_path}")
            print(f"Best Val MAE: {best_val_mae:.2f} months")
            print(
                "Best Bland–Altman 95% Limits of Agreement: "
                f"{best_bland_altman_stats['lower_95_limit_of_agreement']:.2f} to "
                f"{best_bland_altman_stats['upper_95_limit_of_agreement']:.2f} months"
            )
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")

        save_checkpoint(
            checkpoint_path=latest_checkpoint_path,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_val_mae=best_val_mae,
            best_epoch=best_epoch,
            patience_counter=patience_counter,
            history=history,
            config=config,
        )

        print(f"Saved latest checkpoint to: {latest_checkpoint_path}")
        print(
            "Latest Bland–Altman 95% Limits of Agreement: "
            f"{latest_bland_altman_stats['lower_95_limit_of_agreement']:.2f} to "
            f"{latest_bland_altman_stats['upper_95_limit_of_agreement']:.2f} months"
        )

        if patience_counter >= PATIENCE:
            print(
                f"Early stopping triggered. "
                f"Best epoch: {best_epoch}, Best Val MAE: {best_val_mae:.2f} months"
            )
            break

    history_df = pd.DataFrame(history)
    best_predictions_path = plot_dir / "best_val_predictions.csv"

    if best_predictions_path.exists():
        best_predictions_df = pd.read_csv(best_predictions_path)
    elif predictions_df is not None:
        best_predictions_df = predictions_df
    else:
        latest_predictions_path = plot_dir / "latest_val_predictions.csv"
        if latest_predictions_path.exists():
            best_predictions_df = pd.read_csv(latest_predictions_path)
        else:
            print("No prediction file found. Skipping final plots.")
            return

    # Final best plots at the end.
    save_plots(
        history_df=history_df,
        predictions_df=best_predictions_df,
        plot_dir=plot_dir,
        prefix="final_best",
    )

    final_bland_altman_stats = save_bland_altman_plot(
        predictions_df=best_predictions_df,
        plot_dir=plot_dir,
        prefix="final_best",
    )

    age_metrics, gender_metrics = group_metrics(
        preds=best_predictions_df["prediction"].values,
        targets=best_predictions_df["target"].values,
        males=best_predictions_df["male"].values,
    )

    age_metrics.to_csv(plot_dir / "mae_by_age_group.csv", index=False)
    gender_metrics.to_csv(plot_dir / "mae_by_gender.csv", index=False)

    final_overall_metrics = {
        "mae": compute_mae(
            best_predictions_df["prediction"].values,
            best_predictions_df["target"].values,
        ),
        "rmse": compute_rmse(
            best_predictions_df["prediction"].values,
            best_predictions_df["target"].values,
        ),
        "median_absolute_error": compute_median_absolute_error(
            best_predictions_df["prediction"].values,
            best_predictions_df["target"].values,
        ),
        "mean_error_bias": compute_mean_error(
            best_predictions_df["prediction"].values,
            best_predictions_df["target"].values,
        ),
        "bland_altman_bias_mean_error": final_bland_altman_stats["bias_mean_error"],
        "bland_altman_sd_error": final_bland_altman_stats["sd_error"],
        "bland_altman_lower_95_loa": final_bland_altman_stats["lower_95_limit_of_agreement"],
        "bland_altman_upper_95_loa": final_bland_altman_stats["upper_95_limit_of_agreement"],
        "best_epoch": best_epoch,
        "best_val_mae": best_val_mae,
    }

    pd.DataFrame([final_overall_metrics]).to_csv(
        plot_dir / "final_overall_metrics.csv",
        index=False,
    )

    print("\nTraining finished.")
    print(f"Best epoch: {best_epoch}")
    print(f"Best validation MAE: {best_val_mae:.2f} months")
    print(f"Best model saved to: {best_checkpoint_path}")
    print(f"Latest checkpoint saved to: {latest_checkpoint_path}")
    print(f"Plots and logs saved to: {plot_dir}")

    print("\nFinal overall metrics from best validation predictions:")
    print(f"MAE: {final_overall_metrics['mae']:.2f} months")
    print(f"RMSE: {final_overall_metrics['rmse']:.2f} months")
    print(f"Median AE: {final_overall_metrics['median_absolute_error']:.2f} months")
    print(f"Mean Error / Bias: {final_overall_metrics['mean_error_bias']:.2f} months")

    print("\nFinal Bland–Altman statistics:")
    print(f"Bias / Mean Error: {final_bland_altman_stats['bias_mean_error']:.2f} months")
    print(f"SD Error: {final_bland_altman_stats['sd_error']:.2f} months")
    print(
        "95% Limits of Agreement: "
        f"{final_bland_altman_stats['lower_95_limit_of_agreement']:.2f} to "
        f"{final_bland_altman_stats['upper_95_limit_of_agreement']:.2f} months"
    )

    print("\nMAE by age group:")
    print(age_metrics)

    print("\nMAE by gender:")
    print(gender_metrics)


if __name__ == "__main__":
    main()

Using device: cuda
Found image files: 12584
Total CSV samples: 12611
Bone age range: 1 to 228 months
Male distribution:
male
True     6833
False    5778
Name: count, dtype: int64
Missing image IDs saved to: /content/drive/MyDrive/Colab Notebooks/training_plots_efficientnet_b4_512x512/missing_images.csv
First missing IDs: ['1435', '1446', '1814', '2178', '2419', '2934', '3079', '3100', '3991', '4270', '5530', '6232', '6319', '7308', '7893', '8580', '8757', '8940', '9969', '10441']
Samples after image-file filtering: 12584
Train samples: 10696
Validation samples: 1888


Model: tf_efficientnet_b4.ns_jft_in1k
Image size: 512x512 width x height
Effective batch size: 128
Total parameters: 18,467,657
Trainable parameters: 18,467,657
No checkpoint loaded. Starting from scratch.

Epoch 1/100


Train loss: 117.0129 | Train MAE: 120.01 months | Val loss: 103.1892 | Val MAE: 106.18 months | Val RMSE: 114.76 months | Val Median AE: 110.86 months | Val Mean Error: -106.16 months | LR: 4.00e-05
Saved new best model to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_best.pth
Best Val MAE: 106.18 months
Best Bland–Altman 95% Limits of Agreement: -191.62 to -20.70 months
Saved latest checkpoint to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_latest.pth
Latest Bland–Altman 95% Limits of Agreement: -191.62 to -20.70 months

Epoch 2/100


Train loss: 77.0872 | Train MAE: 80.03 months | Val loss: 46.8244 | Val MAE: 49.69 months | Val RMSE: 73.45 months | Val Median AE: 26.44 months | Val Mean Error: -48.06 months | LR: 6.00e-05
Saved new best model to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_best.pth
Best Val MAE: 49.69 months
Best Bland–Altman 95% Limits of Agreement: -156.95 to 60.82 months
Saved latest checkpoint to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_latest.pth
Latest Bland–Altman 95% Limits of Agreement: -156.95 to 60.82 months

Epoch 3/100


Train loss: 35.5448 | Train MAE: 38.38 months | Val loss: 15.0479 | Val MAE: 17.84 months | Val RMSE: 22.93 months | Val Median AE: 15.23 months | Val Mean Error: -15.50 months | LR: 8.00e-05
Saved new best model to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_best.pth
Best Val MAE: 17.84 months
Best Bland–Altman 95% Limits of Agreement: -48.62 to 17.62 months
Saved latest checkpoint to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_latest.pth
Latest Bland–Altman 95% Limits of Agreement: -48.62 to 17.62 months

Epoch 4/100


...

Train loss: 1.3986 | Train MAE: 3.30 months | Val loss: 5.5072 | Val MAE: 8.02 months | Val RMSE: 10.31 months | Val Median AE: 6.62 months | Val Mean Error: -2.40 months | LR: 7.70e-06
No improvement. Patience: 11/15
Saved latest checkpoint to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_latest.pth
Latest Bland–Altman 95% Limits of Agreement: -22.05 to 17.24 months

Epoch 84/100


Train loss: 1.4111 | Train MAE: 3.31 months | Val loss: 5.3203 | Val MAE: 7.83 months | Val RMSE: 10.12 months | Val Median AE: 6.38 months | Val Mean Error: -1.23 months | LR: 6.84e-06
No improvement. Patience: 12/15
Saved latest checkpoint to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_latest.pth
Latest Bland–Altman 95% Limits of Agreement: -20.92 to 18.46 months

Epoch 85/100


Train loss: 1.4331 | Train MAE: 3.33 months | Val loss: 5.2535 | Val MAE: 7.75 months | Val RMSE: 10.05 months | Val Median AE: 6.25 months | Val Mean Error: -0.65 months | LR: 6.03e-06
No improvement. Patience: 13/15
Saved latest checkpoint to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_latest.pth
Latest Bland–Altman 95% Limits of Agreement: -20.32 to 19.02 months

Epoch 86/100


Train loss: 1.4181 | Train MAE: 3.31 months | Val loss: 5.2832 | Val MAE: 7.78 months | Val RMSE: 10.06 months | Val Median AE: 6.38 months | Val Mean Error: -1.21 months | LR: 5.26e-06
No improvement. Patience: 14/15
Saved latest checkpoint to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_latest.pth
Latest Bland–Altman 95% Limits of Agreement: -20.80 to 18.38 months

Epoch 87/100


Train loss: 1.4700 | Train MAE: 3.37 months | Val loss: 5.3264 | Val MAE: 7.83 months | Val RMSE: 10.11 months | Val Median AE: 6.41 months | Val Mean Error: -1.52 months | LR: 4.55e-06
No improvement. Patience: 15/15
Saved latest checkpoint to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_latest.pth
Latest Bland–Altman 95% Limits of Agreement: -21.12 to 18.07 months
Early stopping triggered. Best epoch: 72, Best Val MAE: 7.73 months

Training finished.
Best epoch: 72
Best validation MAE: 7.73 months
Best model saved to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_best.pth
Latest checkpoint saved to: /content/drive/MyDrive/Colab Notebooks/efficientnet_b4_512x512_latest.pth
Plots and logs saved to: /content/drive/MyDrive/Colab Notebooks/training_plots_efficientnet_b4_512x512

Final overall metrics from best validation predictions:
MAE: 7.73 months
RMSE: 10.03 months
Median AE: 6.12 months
Mean Error / Bias: -0.37 months

Final Bland–Altman statistics

In [1]:
import os

print(os.listdir('/content/drive/MyDrive/Colab Notebooks'))
print(os.cpu_count())

['boneage-training-dataset.csv', 'convnextv2_huge_22k_512_ema.pt', 'overlayed_RSNA_dataset', 'training_plots', 'resnet50_best.pth', 'resnet50_train_448.ipynb', 'training_plots_resnet50_448', 'resnet50_448_best.pth', 'training_plots_convnextv2_base_512', 'training_plots_convnextv2_base_384x512', 'train_convnextv2_512x384.ipynb', 'cropped_overlayed_RSNA_dataset', 'training_plots_efficientnet_b4_512x512', 'convnextv2_base_384x512_best.pth', 'resnet50_train.ipynb', 'convnextv2_base_384x512_latest.pth', 'efficientnet_b4_512x512_best.pth', 'efficientnet_b4_512x512_latest.pth', 'train_efficientnetB4_512.ipynb']
12


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi